# Optimize spatio-temporal synthetic-data parameters

Goal: find parameters such that IID, cluster-aware, and spatial TOST conclude equivalence at Δ=1, but the spatio-temporal TOST rejects equivalence at Δ=1.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import os, sys, json
from pathlib import Path

# If this notebook is in the directory that contains `main_module/`, add it to sys.path
# ROOT = Path('.').resolve()
# if str(ROOT) not in sys.path:
#     sys.path.insert(0, str(ROOT))

REPO_ROOT = Path("/Users/dhetting/src/pyTOST")  # adjust if needed
sys.path.insert(0, str(REPO_ROOT))


# print('cwd:', ROOT)

from pyTOST.data_gen.optimize_spatiotemporal_params import search, save_best


/Users/dhetting/src/pyTOST/pyTOST/engines/spatial_tost.py:480: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  covariance parameters :math:`(\sigma^2, \rho, \tau^2)` for each candidate ``mu``.


In [2]:
# Fast-but-real search settings:
# - small-B bootstraps during search keep evaluation per candidate cheap
# - subprocess + timeout prevents pathological ML fits from stalling the whole search
best, hist = search(
    seed=123,
    n_iter=600,                 # upper bound; search stops early if it finds a strong candidate
    alpha=0.05,
    margin=1.0,                 # target equivalence margin for the demo
    hac_lags=8,                 # temporal baseline (Newey–West)
    verbose_every=25,
    # speed controls (still statistically defensible)
    st_param_B=20,              # ST parametric bootstrap reps during search
    st_block_B=20,              # ST time-block bootstrap reps during search
    prescreen_buffer=0.12,      # require ST CI noticeably wider than naive CI before confirming
    use_subprocess=True,
    candidate_timeout_seconds=120,
    debug_stage_prints=False,
)


Pattern found at iter 2: score=5.086


In [3]:
from pprint import pprint

In [4]:
pprint(best.params)

{'delta': 0.9706665755707152,
 'domain': (-2.0, 2.0, -2.0, 2.0),
 'length_scale': 2.557763924844609,
 'n_buildings': 7,
 'n_space': 23,
 'n_time': 76,
 'obs_sd': 0.19413360802462348,
 'rho': 0.8700558526476226,
 'seed': 1124,
 'spatial_sd': 1.024054159986033}


In [12]:
best.params['n_buildings'] = 25

In [5]:
print(f'IID: {best.eq_iid}\nCluster: {best.eq_cluster}\nSpatial: {best.eq_spatial}\nTemporal: {best.eq_temporal}\nSpatiotemporal: {best.eq_spatiotemporal}')

IID: True
Cluster: True
Spatial: True
Temporal: True
Spatiotemporal: False


In [6]:
print(f'IID: {best.ci_iid}\nCluster: {best.ci_cluster}\nSpatial: {best.ci_spatial}\nTemporal: {best.ci_temporal}\nSpatiotemporal: {best.ci_spatiotemporal}')

IID: (0.5042633865744832, 0.5899796238402804)
Cluster: (0.4229825227631768, 0.6712604876515867)
Spatial: (nan, nan)
Temporal: (0.46230171293777494, 0.6319412974769891)
Spatiotemporal: (0.3151324811126705, 1.3477037970927297)


In [14]:
# Save the best generator parameters with a *stronger* (but still reasonable) re-validation.
# Note: this re-runs the winning candidate with larger bootstrap B, which can take a few minutes.
# You can reduce st_param_B / st_block_B, or set st_block_refit_cov=False, to speed this up.
out_path = "./best_spatiotemporal_params_v2.json"
save_best(
    best,
    out_path,
    alpha=0.05,
    margin=1.0,
    hac_lags=8,
    st_param_B=80,
    st_block_B=80,
    st_block_refit_cov=True,
)

# Ensure the saved parameters are *exactly* the kwargs accepted by the data generator,
# plus separate panel-building controls used to construct the diff panel.
import inspect, json
from pyTOST.data_gen import synthetic_tost_data

gen_sig = inspect.signature(synthetic_tost_data.generate_spatiotemporal)
gen_allowed = set(gen_sig.parameters.keys())

with open(out_path, "r", encoding="utf-8") as f:
    payload = json.load(f)

params_all = dict(payload.get("params", {}))
generator_kwargs = {k: v for k, v in params_all.items() if k in gen_allowed}
panel_kwargs = {k: v for k, v in params_all.items() if k not in gen_allowed}

payload["generator_kwargs"] = generator_kwargs
payload["panel_kwargs"] = panel_kwargs

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, sort_keys=True)

print("Wrote", out_path)
print("Generator kwargs keys:", sorted(generator_kwargs.keys()))
print("Panel kwargs keys:", sorted(panel_kwargs.keys()))


Wrote ./best_spatiotemporal_params_v2.json
Generator kwargs keys: ['delta', 'domain', 'length_scale', 'n_space', 'n_time', 'obs_sd', 'rho', 'seed', 'spatial_sd']
Panel kwargs keys: ['n_buildings']


In [10]:
panel_kwargs

{'n_buildings': 7}

In [8]:
import inspect
import pandas as pd

from pyTOST.engines.iid_tost import IIDTOST
from pyTOST.engines.cluster_tost import ClusterTOST
from pyTOST.engines.spatial_tost import SpatialTOST
from pyTOST.engines.spatiotemporal_tost import SpatioTemporalTOST
from pyTOST.data_gen import synthetic_tost_data

# Rebuild a dataset using the best params and report results at Δ in {1,3,5}
params_all = dict(best.params)

# Split generator kwargs (exact signature match) vs panel-construction kwargs.
gen_sig = inspect.signature(synthetic_tost_data.generate_spatiotemporal)
gen_allowed = set(gen_sig.parameters.keys())
gen_kwargs = {k: v for k, v in params_all.items() if k in gen_allowed}
panel_kwargs = {k: v for k, v in params_all.items() if k not in gen_allowed}

n_buildings = int(panel_kwargs.get("n_buildings", 10))

df_long, meta = synthetic_tost_data.generate_spatiotemporal(**gen_kwargs)

# Build the paired-difference panel used by the engines in this demo.
wide = df_long.pivot(index="sample_id", columns="arm", values="y").reset_index()
meta2 = df_long[df_long["arm"] == "A"][["sample_id", "x", "y_sp", "t"]].copy()
meta2 = meta2.merge(wide, on="sample_id")

# Standardize column names expected by engines
df = meta2.rename(columns={"t": "time"}).copy()
df["diff"] = df["B"] - df["A"]
df["building_id"] = df["sample_id"].astype(int) % n_buildings

df.head()


,sample_id,x,y_sp,time,A,B,diff,building_id
0,0,-1.279833,-1.609599,0,0.231523,1.720228,1.488705,0
1,1,-1.279833,-1.609599,1,0.585432,3.118586,2.533153,1
2,2,-1.279833,-1.609599,2,0.286432,3.468680,3.182248,2
3,3,-1.279833,-1.609599,3,-0.781155,2.726034,3.507188,3
4,4,-1.279833,-1.609599,4,-1.374460,3.681377,5.055837,4
